# **American Sign: EfficientNet Classifier w/TPU**
train/valid data not splitted

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers
from tensorflow.keras.applications import MobileNetV2
from kaggle_datasets import KaggleDatasets

In [ ]:
# ============================================================================
# 1. TPU INITIALIZATION
# ============================================================================
def initialize_accelerator():
    """Initialize TPU or fallback to CPU/GPU."""
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(tpu)
        tf.tpu.experimental.initialize_tpu_system(tpu)
        strategy = tf.distribute.TPUStrategy(tpu)
        
        print(f'✅ Running on TPU: {tpu.master()}')
        print(f'🚀 Number of TPU cores: {strategy.num_replicas_in_sync}')
        
    except (ValueError, RuntimeError) as e:
        strategy = tf.distribute.get_strategy()
        
        gpus = tf.config.list_physical_devices('GPU')
        if gpus:
            print(f'ℹ️ Running on GPU: {len(gpus)} device(s) found')
        else:
            print('ℹ️ Running on CPU')
    
    return strategy

In [ ]:
# ============================================================================
# 2. IMPROVED IMAGE CLASSIFIER WITH REDUCED DATA
# ============================================================================
class ReducedDataImageClassifier:
    def __init__(self, data_dir, img_size=(224, 224), strategy=None, max_samples_per_class=100):
        """Initialize classifier with reduced data option."""
        self.data_dir = data_dir
        self.img_size = img_size
        self.strategy = strategy if strategy is not None else tf.distribute.get_strategy()
        self.max_samples_per_class = max_samples_per_class  # Maximum samples per class
        self.class_names = []
        self.model = None
        self.base_model = None
        self.history = None
        self.class_weights = None
                
    def load_and_split_data(self, test_size=0.15, val_size=0.15, random_state=42):
        """Load and split data with reduced samples."""
        data_dir = self.data_dir
        all_paths, all_labels = [], []
        
        self.class_names = sorted([d for d in os.listdir(data_dir) 
                                   if os.path.isdir(os.path.join(data_dir, d))])
        num_classes = len(self.class_names)
        print(f"Found {num_classes} classes: {self.class_names}")
        
        class_counts = {}
        for class_idx, class_name in enumerate(self.class_names):
            class_dir = os.path.join(data_dir, class_name)
            count = 0
            if os.path.exists(class_dir):
                # Get image files
                image_files = []
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                        image_files.append(img_name)
                
                # Limit samples per class
                if len(image_files) > self.max_samples_per_class:
                    # Random sampling
                    np.random.seed(random_state)
                    selected_files = np.random.choice(
                        image_files, 
                        size=self.max_samples_per_class, 
                        replace=False
                    )
                    image_files = selected_files
                    print(f"  {class_name}: Limited from {len(image_files) + self.max_samples_per_class} to {self.max_samples_per_class} samples")
                
                for img_name in image_files:
                    all_paths.append(os.path.join(class_dir, img_name))
                    all_labels.append(class_idx)
                    count += 1
            
            class_counts[class_name] = count
        
        print("\n📊 Class Distribution (Reduced):")
        for name, count in class_counts.items():
            print(f"  {name}: {count} images")
        
        if len(all_paths) == 0:
            raise ValueError("No images found!")
        
        # Train/test split
        train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
            all_paths, all_labels, 
            test_size=test_size, 
            stratify=all_labels, 
            random_state=random_state
        )
        
        # Train/validation split
        val_ratio = val_size / (1 - test_size)
        train_paths, val_paths, train_labels, val_labels = train_test_split(
            train_val_paths, train_val_labels,
            test_size=val_ratio,
            stratify=train_val_labels,
            random_state=random_state
        )
        
        print(f"\n📦 Dataset Splits (Reduced):")
        print(f"  Train: {len(train_paths)} images")
        print(f"  Validation: {len(val_paths)} images")
        print(f"  Test: {len(test_paths)} images")
        
        self.train_paths = train_paths
        self.val_paths = val_paths
        self.test_paths = test_paths
        self.train_labels = train_labels
        self.val_labels = val_labels
        self.test_labels = test_labels
        
        # Compute class weights
        raw_weights = compute_class_weight(
            'balanced',
            classes=np.unique(train_labels),
            y=train_labels
        )
        max_weight = 5.0
        capped_weights = np.minimum(raw_weights, max_weight)
        self.class_weights = dict(enumerate(capped_weights))
        print(f"\n⚖️ Class weights: {self.class_weights}")
        
        return train_paths, val_paths, test_paths, train_labels, val_labels, test_labels, num_classes

    def create_data_generators(self, batch_size=16):
        """Create optimized data generators with better preprocessing."""
        
        def load_and_preprocess(path, label):
            image = tf.io.read_file(path)
            image = tf.image.decode_jpeg(image, channels=3)
            image = tf.image.resize(image, self.img_size)
            # CRITICAL: Use MobileNetV2 preprocessing
            image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
            return image, label
        
        def augment(image, label):
            """More aggressive augmentation for small dataset."""
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_brightness(image, 0.2)
            image = tf.image.random_contrast(image, 0.8, 1.2)
            image = tf.image.random_saturation(image, 0.8, 1.2)
            if tf.random.uniform([]) > 0.5:
                image = tf.image.rot90(image, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
            return image, label
        
        num_classes = len(self.class_names)
        
        # Training dataset
        train_ds = tf.data.Dataset.from_tensor_slices((self.train_paths, self.train_labels))
        train_ds = train_ds.shuffle(1000, reshuffle_each_iteration=True)
        train_ds = train_ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
        train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
        train_ds = train_ds.map(lambda x, y: (x, tf.one_hot(y, num_classes)))
        train_ds = train_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        
        # Validation dataset
        val_ds = tf.data.Dataset.from_tensor_slices((self.val_paths, self.val_labels))
        val_ds = val_ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
        val_ds = val_ds.map(lambda x, y: (x, tf.one_hot(y, num_classes)))
        val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        
        # Test dataset
        test_ds = tf.data.Dataset.from_tensor_slices((self.test_paths, self.test_labels))
        test_ds = test_ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
        test_ds = test_ds.map(lambda x, y: (x, tf.one_hot(y, num_classes)))
        test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        
        self.train_generator = train_ds
        self.val_generator = val_ds
        self.test_generator = test_ds
        
        return train_ds, val_ds, test_ds
    
    def build_model(self, num_classes):
        """Build model with better architecture for small dataset."""
        
        with self.strategy.scope():
            base_model = MobileNetV2(
                weights='imagenet',
                include_top=False,
                input_shape=(*self.img_size, 3)
            )
            
            # CRITICAL FIX: Unfreeze MORE layers for fine-tuning
            base_model.trainable = True
            # Only freeze the first 50 layers (out of 155)
            for layer in base_model.layers[:50]:
                layer.trainable = False
            
            print(f"Total layers: {len(base_model.layers)}")
            print(f"Trainable layers: {sum([1 for l in base_model.layers if l.trainable])}")
            
            inputs = keras.Input(shape=(*self.img_size, 3))
            # CRITICAL: Set training=True for fine-tuning
            x = base_model(inputs, training=True)
            x = layers.GlobalAveragePooling2D()(x)
            x = layers.Dropout(0.3)(x)
            x = layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(0.01))(x)
            x = layers.BatchNormalization()(x)
            x = layers.Dropout(0.4)(x)
            x = layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(0.01))(x)
            x = layers.Dropout(0.3)(x)
            outputs = layers.Dense(num_classes, activation='softmax')(x)
            
            model = keras.Model(inputs, outputs)
            
            # CRITICAL FIX: Use smaller learning rate for fine-tuning
            # Don't scale by replicas - use fixed small LR
            base_lr = 1e-4
            
            model.compile(
                optimizer=optimizers.Adam(learning_rate=base_lr),
                loss='categorical_crossentropy',
                metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=2, name='top_2_accuracy')]
            )
            
            self.model = model
            self.base_model = base_model
            return model

    def train(self, batch_size=16, epochs=50):  # Reduced epochs for faster training
        """Train with two-phase approach (optimized for reduced data)."""
        
        print("=" * 70)
        print("🚀 STARTING TRAINING WITH REDUCED DATA")
        print("=" * 70)
        
        train_paths, val_paths, test_paths, train_labels, val_labels, test_labels, num_classes = self.load_and_split_data()
        self.create_data_generators(batch_size=batch_size)
        model = self.build_model(num_classes)
        
        print(f"\n📋 Model Summary:")
        model.summary()
        
        # Phase 1: Train with frozen base (shorter for small dataset)
        print("\n" + "=" * 70)
        print("PHASE 1: Training classifier head (base frozen)")
        print("=" * 70)
        
        # Freeze ALL base layers
        for layer in self.base_model.layers:
            layer.trainable = False
        
        model.compile(
            optimizer=optimizers.Adam(learning_rate=1e-3),
            loss='categorical_crossentropy',
            metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=2, name='top_2_accuracy')]
        )
        
        history1 = model.fit(
            self.train_generator,
            epochs=10,  # Reduced from 20
            validation_data=self.val_generator,
            class_weight=self.class_weights,
            verbose=1
        )
        
        # Phase 2: Fine-tune with unfrozen layers
        print("\n" + "=" * 70)
        print("PHASE 2: Fine-tuning (unfreezing layers)")
        print("=" * 70)
        
        # Unfreeze last 100 layers
        for layer in self.base_model.layers[-100:]:
            layer.trainable = True
        
        model.compile(
            optimizer=optimizers.Adam(learning_rate=1e-5),  # Much smaller LR
            loss='categorical_crossentropy',
            metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=2, name='top_2_accuracy')]
        )
        
        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor='val_accuracy',
                patience=10,  # Reduced patience for faster training
                restore_best_weights=True,
                verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=1e-7,
                verbose=1
            ),
            keras.callbacks.ModelCheckpoint(
                'best_model_reduced.h5',
                monitor='val_accuracy',
                save_best_only=True,
                verbose=1
            )
        ]
        
        history2 = model.fit(
            self.train_generator,
            epochs=epochs,
            validation_data=self.val_generator,
            callbacks=callbacks,
            class_weight=self.class_weights,
            verbose=1
        )
        
        # Combine histories
        self.history = {
            key: history1.history[key] + history2.history[key]
            for key in history1.history.keys()
        }
        
        print("\n✅ Training completed!")
        return history2
    
    def evaluate_and_report(self):
        """Evaluate model and generate report."""
        print("\n" + "=" * 70)
        print("📊 EVALUATION ON TEST SET")
        print("=" * 70)
        
        if self.model is None:
            print("❌ Model not trained yet!")
            return None
        
        results = self.model.evaluate(self.test_generator, verbose=1)
        print(f"\n📈 Test Results:")
        print(f"  Loss: {results[0]:.4f}")
        print(f"  Accuracy: {results[1]:.4f}")
        print(f"  Top-2 Accuracy: {results[2]:.4f}")
        
        all_predictions = []
        all_true_labels = []
        
        for batch_images, batch_labels in self.test_generator:
            predictions = self.model.predict(batch_images, verbose=0)
            predicted_classes = np.argmax(predictions, axis=1)
            true_classes = np.argmax(batch_labels.numpy(), axis=1)
            all_predictions.extend(predicted_classes)
            all_true_labels.extend(true_classes)
        
        report = classification_report(
            all_true_labels, all_predictions,
            target_names=self.class_names,
            output_dict=True,
            zero_division=0
        )
        
        print(f"\n📋 CLASSIFICATION REPORT:")
        print("=" * 70)
        report_df = pd.DataFrame(report).transpose()
        print(report_df.round(3))
        
        self.plot_confusion_matrix(all_true_labels, all_predictions)
        
        return report
    
    def plot_confusion_matrix(self, true_labels, predictions):
        """Plot confusion matrix."""
        cm = confusion_matrix(true_labels, predictions)
        
        plt.figure(figsize=(12, 10))
        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=self.class_names,
            yticklabels=self.class_names
        )
        plt.title("Confusion Matrix (Reduced Data)", fontsize=16, fontweight='bold')
        plt.ylabel("True Label", fontsize=12)
        plt.xlabel("Predicted Label", fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()
    
    def plot_training_history(self):
        """Plot training curves."""
        if self.history is None:
            print("No history available!")
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        epochs = range(1, len(self.history['accuracy']) + 1)
        
        axes[0, 0].plot(epochs, self.history['accuracy'], 'b-', label='Train', linewidth=2)
        axes[0, 0].plot(epochs, self.history['val_accuracy'], 'r-', label='Validation', linewidth=2)
        axes[0, 0].set_title('Accuracy (Reduced Data)', fontsize=14, fontweight='bold')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Accuracy')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        axes[0, 1].plot(epochs, self.history['loss'], 'b-', label='Train', linewidth=2)
        axes[0, 1].plot(epochs, self.history['val_loss'], 'r-', label='Validation', linewidth=2)
        axes[0, 1].set_title('Loss (Reduced Data)', fontsize=14, fontweight='bold')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        axes[1, 0].plot(epochs, self.history['top_2_accuracy'], 'b-', label='Train', linewidth=2)
        axes[1, 0].plot(epochs, self.history['val_top_2_accuracy'], 'r-', label='Validation', linewidth=2)
        axes[1, 0].set_title('Top-2 Accuracy (Reduced Data)', fontsize=14, fontweight='bold')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Top-2 Accuracy')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        if 'lr' in self.history or 'learning_rate' in self.history:
            lr_key = 'lr' if 'lr' in self.history else 'learning_rate'
            axes[1, 1].plot(epochs, self.history[lr_key], 'g-', linewidth=2)
            axes[1, 1].set_title('Learning Rate', fontsize=14, fontweight='bold')
            axes[1, 1].set_xlabel('Epoch')
            axes[1, 1].set_ylabel('Learning Rate')
            axes[1, 1].set_yscale('log')
            axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        best_val_acc = max(self.history['val_accuracy'])
        best_epoch = self.history['val_accuracy'].index(best_val_acc) + 1
        print(f"\n🏆 Best Validation Accuracy: {best_val_acc:.4f} (Epoch {best_epoch})")

    def save_model(self, filepath="reduced_data_classifier.h5"):
        """Save model."""
        if self.model is not None:
            self.model.save(filepath)
            print(f"💾 Model saved to: {filepath}")
        else:
            print("❌ No model to save!")

In [ ]:
# ============================================================================
# 3. MAIN EXECUTION WITH REDUCED DATA OPTIONS
# ============================================================================
if __name__ == "__main__":
    strategy = initialize_accelerator()

    primary_path = "/kaggle/input/datasets/kapillondhe/american-sign-language/ASL_Dataset/Train"
    fallback_path = "/kaggle/input/american-sign-language/ASL_Dataset/Train"

    if os.path.exists(primary_path):
        data_dir = primary_path
        print(f"✅ Found data at primary path: {data_dir}")
    elif os.path.exists(fallback_path):
        data_dir = fallback_path
        print(f"✅ Found data at fallback path: {data_dir}")
    else:
        try:
            from kaggle_datasets import KaggleDatasets
            ds_name = "american-sign-language"
            data_dir_gcs = KaggleDatasets().get_gcs_path(ds_name)
            data_dir = os.path.join(data_dir_gcs, "ASL_Dataset", "Train")
            print(f"📁 GCS Path resolved: {data_dir}")
        except Exception as e:
            print(f"⚠️ Path detection failed. Please check the Data sidebar.")
            data_dir = primary_path 

    # ============================================================================
    # CHOOSE YOUR DATA REDUCTION STRATEGY
    # ============================================================================

    MAX_SAMPLES_PER_CLASS = 100  # Use only 100 samples per class
    GLOBAL_BATCH_SIZE = 32 
    
    classifier = ReducedDataImageClassifier(
        data_dir=data_dir,
        strategy=strategy,
        img_size=(224, 224),
        max_samples_per_class=MAX_SAMPLES_PER_CLASS  # Add this parameter
    )
    
    try:
        print(f"\n🚀 Starting training with MAX {MAX_SAMPLES_PER_CLASS} samples per class...")
        classifier.train(batch_size=GLOBAL_BATCH_SIZE, epochs=50)  # Reduced epochs
        
        print("\n📊 Evaluating model...")
        classifier.evaluate_and_report()
        
        print("\n📈 Plotting training history...")
        classifier.plot_training_history()
        
        classifier.save_model(f"classifier_{MAX_SAMPLES_PER_CLASS}_samples.h5")
        
        print("\n" + "=" * 70)
        print("✅ ALL DONE! (Reduced Data Version)")
        print("=" * 70)
        
    except Exception as e:
        print(f"❌ Error during execution: {e}")
        import traceback
        traceback.print_exc()